In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder, RobustScaler
from sklearn.metrics import silhouette_score, make_scorer
import joblib
import seaborn as sns
from sklearn.model_selection import GridSearchCV, KFold
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import pygeohash as pgh
import geopy as gpy
import numpy as np
import warnings
from sklearn.decomposition import PCA
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from sklearn.mixture import GaussianMixture
import os

%load_ext autoreload
%autoreload 2

Algorithm: K-Means clustering (unsupervised) to group research centers into three quality tiers (Premium, Standard, Basic) based on infrastructure and healthcare access features. Despite location discrapancy, assume a research center is a different company, so if you see two centres together but different attributes. could be why

In [ ]:
filename = 'research_centers.csv'
columns_to_drop = ['researchCenterId', 'researchCenterName']
geographical_cols = ['latitude', 'longitude']
count_data_cols = ['internalFacilitiesCount', 'hospitals_10km', 'pharmacies_10km']
ratio_cols = ['facilityDiversity_10km', 'facilityDensity_10km']
categorical_cols = ['city']
colors = plt.cm.tab20.colors
geohash_precision = 4
geolocater = gpy.geocoders.Nominatim(user_agent="research_grid_geocoder")
os.makedirs('models', exist_ok=True)

# Load

In [ ]:
raw_df = pd.read_csv(filename)
raw_df.head()

In [ ]:
raw_df.info(), raw_df.duplicated().sum()

# Explore

In [ ]:
raw_df['researchCenterName'].nunique(), raw_df['city'].nunique(), raw_df.columns

In [ ]:
df = raw_df.set_index('researchCenterId')
for geohash_num in range(2, 10):
    df[f'geohash_{geohash_num}'] = df.apply(lambda x: pgh.encode(x['latitude'], x['longitude'], precision = geohash_num), axis=1)
df['external_score'] = df['hospitals_10km'] + df['pharmacies_10km']
df['internal_external_ratio'] = df['internalFacilitiesCount'] / (df['hospitals_10km'] + df['pharmacies_10km'] + 1) 
df['healthcare_deficit'] = (df['hospitals_10km'] + df['pharmacies_10km']) == 0
df['lat_rad'] = np.radians(df['latitude'])
df['lon_rad'] = np.radians(df['longitude'])
df['cart_x'] = np.cos(df['lat_rad']) * np.cos(df['lon_rad'])
df['cart_y'] = np.cos(df['lat_rad']) * np.sin(df['lon_rad'])
df['cart_z'] = np.sin(df['lat_rad'])
df.drop(columns =['lat_rad', 'lon_rad'], inplace=True)
# coordinates_series = df['latitude'].astype(str).str.cat(df['longitude'].astype(str), sep=',') 
df.head()

In [ ]:
df[['city','geohash_2', 'geohash_3', 'geohash_4']].nunique()

In [ ]:
df.groupby('city').agg(**{
    'precision 2 unique': ('geohash_2', 'nunique'),
    'precision 2 names': ('geohash_2', 'unique'),
    'precision 3 unique': ('geohash_3', 'nunique'),
    'precision 3 names': ('geohash_3', 'unique'),
    'precision 4 unique': ('geohash_4', 'nunique'),
    'precision 4 names': ('geohash_4', 'unique')
})

### Research Centers - each record is a research center

In [ ]:
df[['researchCenterName']].nunique()

Filtering needs to be done by examinining and looking at the city heat map

### City

In [ ]:
df['city'].nunique(), df['city'].unique()

In [ ]:
city_summary_df = df.groupby('city').agg(**{
    'Total Research Centers': ('researchCenterName', 'nunique'),
    'Total Facilities': ('internalFacilitiesCount', 'sum'),
    'Total Hospitals': ('hospitals_10km', 'sum'),
    'Total Pharmacies': ('pharmacies_10km', 'sum')
})
city_summary_df.loc['Total', :]  = city_summary_df.sum()
city_summary_df.loc[:,'Research Distribution'] = round(city_summary_df['Total Research Centers'] / city_summary_df.loc['Total', 'Total Research Centers'] * 100, 2)
city_summary_df.loc[:,'Facility Distribution'] = round(city_summary_df['Total Facilities'] / city_summary_df.loc['Total', 'Total Facilities'] * 100, 2)
city_summary_df.loc[:,'Hospital Distribution'] = round(city_summary_df['Total Hospitals'] / city_summary_df.loc['Total', 'Total Hospitals'] * 100, 2)
city_summary_df.loc[:,'Pharmacy Distribution'] = round(city_summary_df['Total Pharmacies'] / city_summary_df.loc['Total', 'Total Pharmacies'] * 100, 2)

city_summary_df

In [ ]:
geohash_summary_list = []

for a_geohash in ['geohash_2', 'geohash_3', 'geohash_4']:
    temp_geohash_summary_df = df.groupby(a_geohash).agg(**{
        'Total Research Centers': ('researchCenterName', 'nunique'),
        'Total Facilities': ('internalFacilitiesCount', 'sum'),
        'Total Hospitals': ('hospitals_10km', 'sum'),
        'Total Pharmacies': ('pharmacies_10km', 'sum')
    })
    temp_geohash_summary_df.loc[f'Total {a_geohash}', :]  = temp_geohash_summary_df.sum()
    temp_geohash_summary_df.loc[:,'Research Distribution'] = round(temp_geohash_summary_df['Total Research Centers'] / temp_geohash_summary_df.loc[f'Total {a_geohash}', 'Total Research Centers'] * 100, 2)
    temp_geohash_summary_df.loc[:,'Facility Distribution'] = round(temp_geohash_summary_df['Total Facilities'] / temp_geohash_summary_df.loc[f'Total {a_geohash}', 'Total Facilities'] * 100, 2)
    temp_geohash_summary_df.loc[:,'Hospital Distribution'] = round(temp_geohash_summary_df['Total Hospitals'] / temp_geohash_summary_df.loc[f'Total {a_geohash}', 'Total Hospitals'] * 100, 2)
    temp_geohash_summary_df.loc[:,'Pharmacy Distribution'] = round(temp_geohash_summary_df['Total Pharmacies'] / temp_geohash_summary_df.loc[f'Total {a_geohash}', 'Total Pharmacies'] * 100, 2)

    temp_geohash_summary_df.drop(index=f'Total {a_geohash}', inplace=True)
    temp_geohash_summary_df = pd.concat([temp_geohash_summary_df.rename_axis('city')], axis=0, keys=[a_geohash])

    geohash_summary_list.append(temp_geohash_summary_df)

geohash_summary_df = pd.concat(geohash_summary_list, axis=0)
geohash_summary_df.head()

In [ ]:
fig = px.scatter(
    df,
    x="latitude",
    y="longitude",
    color="city",
    hover_data=["researchCenterName"]
)

fig.update_layout(title="City Locations by Coordinates")
fig.show()

In [ ]:
fig = px.scatter(
    df,
    x="latitude",
    y="longitude",
    color="geohash_2",
    hover_data=["researchCenterName"]
)

fig.update_layout(title="City Locations by Coordinates")
fig.show()

In [ ]:
fig = px.scatter(
    df,
    x="latitude",
    y="longitude",
    color="geohash_3",
    hover_data=["researchCenterName"]
)

fig.update_layout(title="City Locations by Coordinates")
fig.show()

In [ ]:
geohash_summary_df.loc['geohash_3']

In [ ]:
fig = px.scatter(
    df,
    x="latitude",
    y="longitude",
    color="geohash_4",
    hover_data=["researchCenterName"]
)

fig.update_layout(title="City Locations by Coordinates")
fig.show()

### Internal Facilities Count

In [ ]:
facility_summary_df = df.groupby('city').agg(**{
    'Average Facilities': ('internalFacilitiesCount', 'mean'),
    'Total Facilities': ('internalFacilitiesCount', 'sum'),
    'Number of Research Centers': ('researchCenterName', 'nunique')
})
facility_summary_df

In [ ]:
geohash_facility_summary_df = df.groupby('geohash_3').agg(**{
    'Average Facilities': ('internalFacilitiesCount', 'mean'),
    'Total Facilities': ('internalFacilitiesCount', 'sum'),
    'Number of Research Centers': ('researchCenterName', 'nunique')
})
geohash_facility_summary_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
facility_summary_df.plot(kind='bar', 
                         y='Total Facilities', 
                         legend=False, 
                         color=colors[:len(facility_summary_df)],
                         ax=ax)
ax.set_title('Total Facilities by City')
ax.set_xlabel('City')
ax.tick_params(axis='x', rotation=0)
ax.set_ylabel('Total Facilities')
ax.grid(axis='y', linestyle='--', alpha=0.7)

In [ ]:
df['internalFacilitiesCount'].hist(bins=11, color='skyblue', edgecolor='black')
plt.title('Distribution of Internal Facilities Count')
plt.xlabel('Internal Facilities Count')
plt.ylabel('Frequency')

In [ ]:
df['internalFacilitiesCount'].plot(kind='box', vert=False, color='skyblue')
plt.title('Box Plot of Internal Facilities Count')
plt.xlabel('All Research Centers')

In [ ]:
cities = df['city'].sort_values().unique()
max_bin_count = df['internalFacilitiesCount'].max()

fig, axes = plt.subplots(3, 2, figsize=(12, 10), sharex=False)
axes = axes.flatten()

for i, city in enumerate(cities):
    subset = df.loc[df['city'] == city, 'internalFacilitiesCount']
    subset.plot(
        kind='hist',
        bins=max_bin_count,
        ax=axes[i],
        edgecolor='black',
        alpha=0.7,
        title=city,
        sharex=False,
        sharey=True
    )
    axes[i].set_xlabel('Facility Count')
    axes[i].set_ylabel('Frequency')
    axes[i].set_xlim(0, max_bin_count + 1)
    axes[i].set_ylim(0, 4)

for j in range(len(cities), len(axes)):
    axes[j].axis('off')

fig.suptitle('internalFacilitiesCount Distribution by City (3x2 grid)', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
cities = df['geohash_3'].sort_values().unique()
max_bin_count = df['internalFacilitiesCount'].max()

fig, axes = plt.subplots(3, 2, figsize=(12, 10), sharex=False)
axes = axes.flatten()

for i, city in enumerate(cities):
    subset = df.loc[df['geohash_3'] == city, 'internalFacilitiesCount']
    subset.plot(
        kind='hist',
        bins=max_bin_count,
        ax=axes[i],
        edgecolor='black',
        alpha=0.7,
        title=city,
        sharex=False,
        sharey=True
    )
    axes[i].set_xlabel('Facility Count')
    axes[i].set_ylabel('Frequency')
    axes[i].set_xlim(0, max_bin_count + 1)
    axes[i].set_ylim(0, 4)

for j in range(len(cities), len(axes)):
    axes[j].axis('off')

fig.suptitle('internalFacilitiesCount Distribution by City (3x2 grid)', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
df.boxplot(column='internalFacilitiesCount', by='city', vert=False, figsize=(10, 6))
plt.title('Box Plot of Internal Facilities Count by City')
plt.suptitle('')

In [ ]:
df.boxplot(column='internalFacilitiesCount', by='geohash_3', vert=False, figsize=(10, 6))
plt.title('Box Plot of Internal Facilities Count by Geohash')
plt.suptitle('')

### Hospital and pharmacies

In [ ]:
df.groupby('city').agg(**{
    'Total Hospitals': ('hospitals_10km', 'sum'),
    'Total Pharmacies': ('pharmacies_10km', 'sum')
})

In [ ]:
df.plot.box(column=['hospitals_10km', 'pharmacies_10km'], by='city', vert=False, figsize=(16, 8))

In [ ]:
df.plot.box(column=['hospitals_10km', 'pharmacies_10km'], by='geohash_3', vert=False, figsize=(16, 8))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6), ncols=2, nrows=1)
sns.scatterplot(
    data=df, 
    x='hospitals_10km', 
    y='internalFacilitiesCount', 
    hue='city',          
    palette='rocket', 
    alpha=0.75,
    ax=ax[0]
)
sns.scatterplot(
    data=df, 
    x='hospitals_10km', 
    y='internalFacilitiesCount', 
    hue='geohash_3',          
    palette='rocket', 
    alpha=0.75,
    ax=ax[1]
)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df, 
    x='hospitals_10km', 
    y='pharmacies_10km', 
    hue='city',          
    palette='rocket', 
    alpha=0.75
)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df, 
    x='hospitals_10km', 
    y='facilityDiversity_10km', 
    hue='city',          
    palette='viridis', 
    alpha=0.75
)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df, 
    x='hospitals_10km', 
    y='facilityDensity_10km', 
    hue='city',          
    palette='rocket', 
    alpha=0.75
)
plt.show()

### Facility Diversity

In [ ]:
df['facilityDiversity_10km'].plot(kind='box', vert=False, color='skyblue')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), ncols=2, nrows=1)
df.plot(kind='box', column='facilityDiversity_10km', by='city', vert=False, figsize=(14, 12), subplots=True, ax=ax[0])
df.plot(kind='box', column='facilityDiversity_10km', by='geohash_3', vert=False, figsize=(14, 12), subplots=True, ax=ax[1])

### Facility Density

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), ncols=2, nrows=1)
df.plot(kind='box', column='facilityDensity_10km', by='city', vert=False, figsize=(14, 12), ax=ax[0])
df.plot(kind='box', column='facilityDensity_10km', by='geohash_3', vert=False, figsize=(14, 12), ax=ax[1])

### Correlation

In [ ]:
df.head()

In [ ]:
numeric_cols = ['internalFacilitiesCount','hospitals_10km','pharmacies_10km','facilityDiversity_10km','facilityDensity_10km', 'external_score', 'internal_external_ratio','cart_x', 'cart_y', 'cart_z', 'healthcare_deficit']
correlation = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation heatmap of key numeric features')
plt.show()

corr_with_diversity = correlation['facilityDiversity_10km'].sort_values(ascending=False)
corr_with_internal = correlation['internalFacilitiesCount'].sort_values(ascending=False)
print('Correlation with facilityDiversity_10km:\n', corr_with_diversity)
print('\nCorrelation with internalFacilitiesCount:\n', corr_with_internal)

# Feature Selection, Preprocessing, and K-Means Clustering

We will use scikit-learn transformers to build a pipeline that standardizes relevant numeric features and fits K-Means. This is purely unsupervised for now (no Gold labels), then we map clusters to `Premium`, `Standard`, `Basic` by center quality signal.

### Baseline Model

In [ ]:
baseline_preprocessor = ColumnTransformer(
    transformers=[
        ('geographical', StandardScaler(), ['latitude', 'longitude']),
        ('categorical', OneHotEncoder(), ['city']),
        ('numerical', StandardScaler(), ['internalFacilitiesCount', 'hospitals_10km', 'pharmacies_10km', 'facilityDiversity_10km', 'facilityDensity_10km'])
    ],
    remainder='drop'
)

X_reference = baseline_preprocessor.fit_transform(df)

results = []
random_labels = np.random.randint(0, 3, size=len(df))
score = silhouette_score(X_reference, random_labels)
results.append({'algorithm': 'Random Assignment', 'silhouette': score})

gmm = GaussianMixture(n_components=3, random_state=42)
labels = gmm.fit_predict(X_reference)
score = silhouette_score(X_reference, labels)
results.append({'algorithm': 'GMM', 'silhouette': score})

baseline_df = pd.DataFrame(results)
baseline_df

### Phase 2

In [ ]:
latitude_longitude_preprocessor = ColumnTransformer(
    transformers=[
        ('geographical', StandardScaler(), ['latitude', 'longitude']),
        ('numerical', StandardScaler(), ['internalFacilitiesCount', 'hospitals_10km', 'pharmacies_10km', 'facilityDiversity_10km', 'facilityDensity_10km', 'external_score', 'internal_external_ratio'])
    ],
    remainder='drop'
)

cart_preprocessor = ColumnTransformer(
    transformers=[
        ('numerical_geo', StandardScaler(), ['cart_x', 'cart_y', 'cart_z']),
        ('numerical', StandardScaler(), ['internalFacilitiesCount', 'hospitals_10km', 'pharmacies_10km', 'facilityDiversity_10km', 'facilityDensity_10km', 'external_score', 'internal_external_ratio'])
    ],
    remainder='drop'
)

geocache_preprocessor = ColumnTransformer(
    transformers=[
        ('categorical', OneHotEncoder(), ['geohash_3']),
        ('numerical', StandardScaler(), ['internalFacilitiesCount', 'hospitals_10km', 'pharmacies_10km', 'facilityDiversity_10km', 'facilityDensity_10km', 'external_score', 'internal_external_ratio'])
    ],
        remainder='drop'
)

experiments = {
    'naive': baseline_preprocessor,
    'coords_only': latitude_longitude_preprocessor,
    'cartesian': cart_preprocessor,
    'geohash': geocache_preprocessor
}

results = []
for name, transformer in experiments.items():
    X_processed = transformer.fit_transform(df)

    for k in range(2, 10):
        model = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = model.fit_predict(X_processed)
        score = silhouette_score(X_processed, labels)
        
        results.append({'experiment': name, 'k': k, 'silhouette': score, 'inertia': model.inertia_})

phase_2_exp = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(10, 6), ncols = 2, nrows=1)
exp_pivot_df = phase_2_exp.pivot(index='k', columns='experiment', values='silhouette')
exp_pivot_inertia_df = phase_2_exp.pivot(index='k', columns='experiment', values='inertia')
exp_pivot_df.plot(marker='o', ax=ax[0])
exp_pivot_inertia_df.plot(marker='o', ax=ax[1])
ax[0].set_title('Silhouette Scores by Experiment')
ax[0].set_xlabel('Number of Clusters (k)')
ax[0].set_ylabel('Silhouette Score')
ax[1].set_title('Inertia by Experiment')
ax[1].set_xlabel('Number of Clusters (k)')
ax[1].set_ylabel('Inertia')
plt.tight_layout()

In [ ]:
geo_cols = ['latitude', 'longitude', 'city', 'geohash_2', 'geohash_3', 'geohash_4', 'geohash_5', 'geohash_6', 'geohash_7', 'geohash_8', 'geohash_9', 'cart_x', 'cart_y', 'cart_z']
num_cols = ['internalFacilitiesCount', 'hospitals_10km', 'pharmacies_10km', 'facilityDiversity_10km', 'facilityDensity_10km', 'external_score', 'internal_external_ratio']

experiments = {
    'baseline': baseline_preprocessor,
    'simple_geohash': geocache_preprocessor,
}

for geocache_num in range(2, 10):
    for pca_num in range(2, 10):
        target_col = f'geohash_{geocache_num}'
        to_drop = ['researchCenterName'] + [c for c in geo_cols if c != target_col]
        
        base_transformer = ColumnTransformer([
            ('drop', 'drop', to_drop),
            ('geo_cat', OneHotEncoder(), [target_col]),
            ('numerical', StandardScaler(), num_cols)
        ])
        
        experiments[f'geohash_{geocache_num}_pca_{pca_num}'] = Pipeline([
            ('prep', base_transformer),
            ('pca', PCA(n_components=pca_num, random_state=42))
        ])

results = []
for name, transformer in experiments.items():
    X_processed = transformer.fit_transform(df)

    for k in range(2, 10):
        model = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = model.fit_predict(X_processed)
        score = silhouette_score(X_processed, labels)
        
        results.append({'experiment': name, 'k': k, 'silhouette': score, 'inertia': model.inertia_})

phase_3_exp = pd.DataFrame(results)
phase_3_exp[phase_3_exp['k'] == 3].sort_values(by='silhouette', ascending=False).head()

In [ ]:
exp_pivot_df = phase_3_exp.pivot(index='k', columns='experiment', values='silhouette')
exp_pivot_inertia_df = phase_3_exp.pivot(index='k', columns='experiment', values='inertia')

experiments_list = list(exp_pivot_df.columns)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Silhouette Scores by Experiment', 'Inertia by Experiment'),
    horizontal_spacing=0.12
)

for experiment in experiments_list:
    fig.add_trace(
        go.Scatter(
            x=exp_pivot_df.index,
            y=exp_pivot_df[experiment],
            mode='lines+markers',
            name=experiment,
            legendgroup=experiment,
            visible=True,
            line=dict(width=2),
            marker=dict(size=8),
            hovertemplate=f'<b>{experiment}</b><br>k=%{{x}}<br>Silhouette=%{{y:.3f}}<extra></extra>'
        ),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Scatter(
            x=exp_pivot_inertia_df.index,
            y=exp_pivot_inertia_df[experiment],
            mode='lines+markers',
            name=experiment,
            legendgroup=experiment,
            showlegend=False,
            visible=True,
            line=dict(width=2),
            marker=dict(size=8),
            hovertemplate=f'<b>{experiment}</b><br>k=%{{x}}<br>Inertia=%{{y:.0f}}<extra></extra>'
        ),
        row=1, col=2
    )

fig.update_layout(
    title_text="Clustering Evaluation Metrics",
    height=500,
    width=1200,
    hovermode='closest',
    legend=dict(
        title="Experiments",
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=1.02,
        bgcolor="rgba(255, 255, 255, 0.8)",
        bordercolor="black",
        borderwidth=1
    )
)

fig.update_xaxes(title_text="Number of Clusters (k)", row=1, col=1)
fig.update_xaxes(title_text="Number of Clusters (k)", row=1, col=2)
fig.update_yaxes(title_text="Silhouette Score", row=1, col=1)
fig.update_yaxes(title_text="Inertia", row=1, col=2)

dropdown_buttons = []
for experiment in experiments_list:
    visibility = []
    for exp in experiments_list:
        visibility.append(exp == experiment)
    visibility = visibility + visibility
    
    dropdown_buttons.append(
        dict(
            label=experiment,
            method="update",
            args=[{"visible": visibility}]
        )
    )

# Add "Show All" button
all_visible = [True] * (len(experiments_list) * 2)
dropdown_buttons.insert(0, dict(
    label="Show All",
    method="update",
    args=[{"visible": all_visible}]
))

fig.update_layout(
    updatemenus=[
        dict(
            buttons=dropdown_buttons,
            direction="down",
            showactive=True,
            x=0.02,
            y=1.15,
            xanchor="left",
            yanchor="top",
            bgcolor="white",
            bordercolor="black",
            borderwidth=1,
            font=dict(size=12)
        )
    ]
)

# Show the plot
fig.show()

# Optionally save as HTML
# fig.write_html("clustering_metrics.html")

# Analysis

In [ ]:
# =========================
# FINAL MODEL (KMEANS MVP)
# =========================
mvp_exp = phase_3_exp[phase_3_exp['k'] == 3] \
    .sort_values(by='silhouette', ascending=False) \
    .reset_index(drop=True).loc[0, 'experiment']

final_pipeline = Pipeline([
    ('preprocessor', experiments[mvp_exp]),
    ('kmeans', KMeans(n_clusters=3, random_state=42, n_init=10))
])

final_pipeline.fit(df)

df_kmeans = df.copy()
df_kmeans['cluster'] = final_pipeline.named_steps['kmeans'].labels_

# Quality score
def compute_quality(df_):
    return (
        df_['internalFacilitiesCount'] * 0.3 +
        df_['hospitals_10km'] * 0.2 +
        df_['pharmacies_10km'] * 0.15 +
        df_['facilityDiversity_10km'] * 0.2 +
        df_['facilityDensity_10km'] * 0.15
    )

df_kmeans['quality_score'] = compute_quality(df_kmeans)

# Map clusters → tiers
tiers = ['Basic', 'Standard', 'Premium']
cluster_quality = df_kmeans.groupby('cluster')['quality_score'].mean().sort_values()

mapping_kmeans = {}
for i, (cluster, _) in enumerate(cluster_quality.items()):
    mapping_kmeans[cluster] = tiers[i]

df_kmeans['qualityTier'] = df_kmeans['cluster'].map(mapping_kmeans)
df_kmeans['model'] = 'KMeans (MVP)'


# =========================
# BASELINE (GMM)
# =========================
baseline_preprocessor = ColumnTransformer(
    transformers=[
        ('geo', StandardScaler(), ['latitude', 'longitude']),
        ('cat', OneHotEncoder(), ['city']),
        ('num', StandardScaler(), [
            'internalFacilitiesCount', 'hospitals_10km', 'pharmacies_10km',
            'facilityDiversity_10km', 'facilityDensity_10km'
        ])
    ]
)

X_ref = baseline_preprocessor.fit_transform(df)

df_gmm = df.copy()
gmm = GaussianMixture(n_components=3, random_state=42)
df_gmm['cluster'] = gmm.fit_predict(X_ref)

df_gmm['quality_score'] = compute_quality(df_gmm)

cluster_quality_gmm = df_gmm.groupby('cluster')['quality_score'].mean().sort_values()

mapping_gmm = {}
for i, (cluster, _) in enumerate(cluster_quality_gmm.items()):
    mapping_gmm[cluster] = tiers[i]

df_gmm['qualityTier'] = df_gmm['cluster'].map(mapping_gmm)
df_gmm['model'] = 'Baseline (GMM)'


# =========================
# RANDOM BASELINE
# =========================
df_random = df.copy()

np.random.seed(42)
df_random['cluster'] = np.random.randint(0, 3, size=len(df_random))

df_random['quality_score'] = compute_quality(df_random)

cluster_quality_rand = df_random.groupby('cluster')['quality_score'].mean().sort_values()

mapping_rand = {}
for i, (cluster, _) in enumerate(cluster_quality_rand.items()):
    mapping_rand[cluster] = tiers[i]

df_random['qualityTier'] = df_random['cluster'].map(mapping_rand)
df_random['model'] = 'Random'


# =========================
# COMBINE ALL
# =========================
combined_df = pd.concat([df_kmeans, df_gmm, df_random], ignore_index=True)


# =========================
# BOXPLOTS (2 COLUMNS GRID)
# =========================
features = [
    'internalFacilitiesCount',
    'hospitals_10km',
    'pharmacies_10km',
    'facilityDiversity_10km',
    'facilityDensity_10km'
]

n_plots = len(features) + 1  # +1 for count plot
n_cols = 2
n_rows = int(np.ceil(n_plots / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 5 * n_rows))
axes = axes.flatten()

for i, feature in enumerate(features):
    sns.boxplot(
        data=combined_df,
        x='qualityTier',
        y=feature,
        hue='model',
        order=['Premium', 'Standard', 'Basic'],
        ax=axes[i]
    )
    axes[i].set_title(f'{feature} by Tier')

# Count plot
tier_counts = combined_df.groupby(['qualityTier', 'model']).size().unstack()
tier_counts.plot(kind='bar', ax=axes[len(features)])
axes[len(features)].set_title('Number of Centers per Tier')
axes[len(features)].set_ylabel('Count')

# Clean legends
for ax in axes:
    if ax.get_legend():
        ax.get_legend().remove()

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper right')

plt.tight_layout()
plt.show()


# =========================
# SCATTER (ONLY MVP)
# =========================
plt.figure(figsize=(8, 6))

scatter = plt.scatter(
    df_kmeans['internalFacilitiesCount'],
    df_kmeans['hospitals_10km'],
    c=df_kmeans['qualityTier'].map({'Premium': 2, 'Standard': 1, 'Basic': 0}),
    cmap='RdYlGn',
    alpha=0.7,
    s=100
)

plt.xlabel('Internal Facilities Count')
plt.ylabel('Hospitals within 10km')
plt.title('MVP (KMeans): Internal Facilities vs Hospitals')

plt.colorbar(scatter, label='Tier (2=Premium, 1=Standard, 0=Basic)')
plt.grid(True, alpha=0.3)
plt.show()